# Video Tokenizer Training (ST VQVAE)

In [ ]:
import torch 

from torch.utils.data import DataLoader
from torchvision.datasets import UCF101

import lightning as L

# Import the model Arch from /models

In [ ]:
from spacetime.models.tokenizers import STVQVae

## Trainer + Objectives

We will use pytorch lightning to reduce boiler plate (there's a lot in previous notebooks, despite the centralized modules in `/src`)

In [ ]:
class STVQVaeModule(L.LightningModule):
    def __init__(
        self,
        num_heads,
        d_model,
        num_layers,
        d_linear,
        codebook_size,
        latent_dim,
        patch_size,
        frame_height,
        frame_width,
        num_frames,
        num_linear_layers=2,
        num_groups=8,
        dropout=0.1,
        beta=0.25
    ):
        super().__init__()
        self.model = STVQVae(
            num_heads=num_heads,
            d_model=d_model,
            num_layers=num_layers,
            d_linear=d_linear,
            codebook_size=codebook_size,
            latent_dim=latent_dim,
            patch_size=patch_size,
            frame_height=frame_height,
            frame_width=frame_width,
            num_frames=num_frames,
            num_linear_layers=num_linear_layers,
            num_groups=num_groups,
            dropout=dropout
        )
        self.beta = beta

    def forward(self, inputs):
        return self.model(inputs)

    def training_step(self, batch, batch_idx):
        x, _ = batch
        
        x_pred, z_e, z_quantized = self(x)
        recon_loss = torch.nn.functional.mse_loss(x_pred, x)
        codebook_loss = torch.nn.functional.mse_loss(z_quantized, z_e.detach())
        commit_loss = torch.nn.functional.mse_loss(z_e, z_quantized.detach())
        loss = recon_loss + codebook_loss + (self.beta * commit_loss)
        
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, _ = batch
        
        x_pred, z_e, z_quantized = self(x)
        recon_loss = torch.nn.functional.mse_loss(x_pred, x)
        codebook_loss = torch.nn.functional.mse_loss(z_quantized, z_e.detach())
        commit_loss = torch.nn.functional.mse_loss(z_e, z_quantized.detach())
        loss = recon_loss + codebook_loss + (self.beta * commit_loss)
        
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.model.parameters(), lr=0.01)

## Load UCF101 Action Recognition dataset 

We use the UCF101 dataset which contains 13,320 videos from 101 action categories. This dataset is commonly used for benchmarking video action recognition models, such as basketball shooting, biking, diving, golf swinging, horse riding, and playing musical instruments.


In [ ]:
def filter_dataset_by_classes(dataset, selected_classes):
    """
    Filter dataset to only include selected classes
    """
    class_to_idx_map = {cls: idx for idx, cls in enumerate(dataset.classes)}
    selected_indices = [class_to_idx_map[cls] for cls in selected_classes if cls in class_to_idx_map]
    
    filtered_samples = [
        (video_path, selected_indices.index(class_idx))
        for video_path, class_idx in dataset.samples
        if class_idx in selected_indices
    ]

    dataset.samples = filtered_samples
    dataset.classes = selected_classes
    return dataset

train_dataset = UCF101(
    root='./data/UCF-101',
    annotation_path='./data/ucfTrainTestlist',
    frames_per_clip=8,
    step_between_clips=8,  # non overlapping clips
    train=True,
)

test_dataset = UCF101(
    root='./data/UCF-101',
    annotation_path='./data/ucfTrainTestlist',
    frames_per_clip=8,
    step_between_clips=8,
    train=False,
)

# Select only a few classes for faster experimentation
selected_classes = [
    'ApplyEyeMakeup', 'ApplyLipstick', 'Archery', 'BabyCrawling', 'BalanceBeam',
    'BandMarching', 'BaseballPitch', 'Basketball', 'BasketballDunk', 'BenchPress'
]

train_dataset = filter_dataset_by_classes(train_dataset, selected_classes)
test_dataset = filter_dataset_by_classes(test_dataset, selected_classes)

print(f"Filtered train dataset size: {len(train_dataset)}")
print(f"Filtered test dataset size: {len(test_dataset)}")

In [14]:
import torch.nn.functional as F


def collate_ucf101(batch):
    # batch: list of (video, label, index) where label is detection labels 
    # and index is the index of the class for recognition
    xs, ys = [], []
    for v, _, l in batch:
        # v: T, H, W, C  (uint8)
        v = v.permute(0, 3, 1, 2)            # -> T, C, H, W
        v = v.float() / 255.0
        v = F.interpolate(v, size=(224, 224), mode='bilinear', align_corners=False)  # resize frames
        v = v.permute(1, 0, 2, 3).contiguous()  # -> C, F, H, W
        xs.append(v.clone())                  # new storage
        ys.append(int(l))
    return torch.stack(xs, 0), torch.tensor(ys, dtype=torch.long)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=8,
    collate_fn=collate_ucf101,
    pin_memory=True,
)

test_dataloader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_ucf101)

In [ ]:
lightning_timesformer = STVQVaeModule(num_classes=101, num_heads=4, d_model=512, d_mlp=512)

trainer = L.Trainer(max_epochs=1, precision=16, fast_dev_run=True)
trainer.fit(model=lightning_timesformer, train_dataloaders=train_dataloader)